# 04A. Alpha Construction Engine

Convert final research-approved signals from Notebook 3F into clean alpha candidates. This notebook creates alpha candidates only; it does not run stress tests, survivor freeze, portfolio construction, or ML.


## 1. Purpose and scope

Build three transparent alpha candidates from approved signal-horizon components: equal-weight, health-weighted, and regime-aware. Existing signal formulas and upstream diagnostic logic are used as-is.


## 2. Imports/config


In [1]:
from __future__ import annotations

import gc
import sqlite3
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.alpha_construction import (
    build_alpha_candidates,
    build_alpha_construction_diagnostics,
    build_alpha_construction_metadata,
    build_alpha_construction_quality,
    build_alpha_correlation_matrix,
    build_alpha_signal_pool,
    build_dynamic_universe_eligibility_mask,
    build_normalized_signal_panels,
    get_approved_alpha_research_signals,
    get_watchlist_diversifier_signals,
    load_alpha_construction_inputs,
    load_candidate_signals_for_names,
)
from src.alpha_construction_storage import (
    ALPHA_CONSTRUCTION_TABLES,
    save_alpha_construction_outputs,
)
from src.db import get_db_path, load_table, table_exists
from src.run_config import make_run_id, make_run_timestamp

ALPHA_CONSTRUCTION_VERSION = 'phase4a_alpha_construction_v4'
SMOOTHING_WINDOW = 10
REBALANCE_FREQUENCY = 5
TURNOVER_CONTROL_ENABLED = True
TURNOVER_CONTROL_UPDATE_RATE = 0.10
WARMUP_TRADING_DAYS = 60
sqlite_db_path = get_db_path()

print(f'SQLite database: {sqlite_db_path}')
print(f'Alpha construction version: {ALPHA_CONSTRUCTION_VERSION}')
print(f'Smoothing window: {SMOOTHING_WINDOW}')
print(f'Rebalance frequency: {REBALANCE_FREQUENCY}')
print(f'Turnover control enabled: {TURNOVER_CONTROL_ENABLED}')
print(f'Turnover control update rate: {TURNOVER_CONTROL_UPDATE_RATE}')
print(f'Coverage warmup trading days: {WARMUP_TRADING_DAYS}')


SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Alpha construction version: phase4a_alpha_construction_v4
Smoothing window: 10
Rebalance frequency: 5
Turnover control enabled: True
Turnover control update rate: 0.1
Coverage warmup trading days: 60


## 3. Create run_id/timestamp


In [2]:
run_id = make_run_id('phase4a_alpha_construction')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')


run_id: phase4a_alpha_construction_20260511_075950
run_timestamp: 2026-05-11 07:59:50


## 4. Load approved research signals from 3F


In [3]:
inputs = load_alpha_construction_inputs(db_path=sqlite_db_path, include_candidate_signals=False)
approved_input_signals = get_approved_alpha_research_signals(
    reproducibility_gate=inputs['reproducibility_gate'],
    signal_health=inputs['signal_health'],
    regime_opportunity=inputs['regime_opportunity'],
)
watchlist_diversifier_signals = get_watchlist_diversifier_signals(inputs['signal_health'])
alpha_signal_pool = build_alpha_signal_pool(
    reproducibility_gate=inputs['reproducibility_gate'],
    signal_health=inputs['signal_health'],
    signal_decay=inputs['signal_decay'],
    regime_opportunity=inputs['regime_opportunity'],
    diversity_selection=inputs['diversity_selection'],
    diversity_similarity=inputs['diversity_similarity'],
)
pool_watchlist_diversifiers = alpha_signal_pool.loc[
    alpha_signal_pool['source_role'].eq('WATCHLIST_DIVERSIFIER')
].copy()

print(f'Approved alpha research signal-horizon components: {len(approved_input_signals)}')
print(f'Watchlist diversifier components selected: {len(watchlist_diversifier_signals)}')
print(f'Alpha signal pool components: {len(alpha_signal_pool)}')
display(approved_input_signals)
display(watchlist_diversifier_signals)
display(alpha_signal_pool)


Approved alpha research signal-horizon components: 0
Watchlist diversifier components selected: 3
Alpha signal pool components: 4


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,...,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,signal_health_score,signal_health_gate,health_notes,run_id,health_version,component_id,source_role
0,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,0.015976,0.015976,0.128788,WATCHLIST,STABLE,...,None,None,None,62.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1,vol_of_vol_20_h10,WATCHLIST_DIVERSIFIER
1,range_expansion_failure_5,20,volatility_structure,POSITIVE_EDGE,WEAK,0.014333,0.014333,0.112310,WATCHLIST,STABLE,...,None,None,None,58.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1,range_expansion_failure_5_h20,WATCHLIST_DIVERSIFIER
2,residual_return_vs_universe_20,20,cross_sectional_relative_value,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK,-0.013551,0.013551,-0.077233,WATCHLIST,STABLE,...,None,None,None,54.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1,residual_return_vs_universe_20_h20,WATCHLIST_DIVERSIFIER


,component_id,signal_name,horizon,signal_family,signal_direction,signal_strength,source_role,signal_health_score,final_research_gate,reproducibility_status,...,best_regime_value,adjusted_best_abs_ic,selected_flag,diversity_group,diversity_candidate_tier,signal_source,orthogonal_version,pool_weight_base,pool_eligible_flag,pool_reason
0,vol_of_vol_20_h10,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,DIVERSITY_SELECTED,62.0,WATCHLIST_ALPHA_RESEARCH,CONDITIONAL_PASS,...,DOWNTREND,0.062676,1,ORTHOGONAL_DIVERSIFIER_SELECTED,ORTHOGONAL_DIVERSIFIER,orthogonal_generated,phase2_orthogonal_signals_v2,0.294920,1,Selected by 03G diversity engine.
1,range_expansion_failure_5_h20,range_expansion_failure_5,20,volatility_structure,POSITIVE_EDGE,WEAK,WATCHLIST_DIVERSIFIER,58.0,NaN,NaN,...,HIGH_VOL,0.031891,0,NaN,NaN,NaN,NaN,0.287236,1,Watchlist diversifier with stable decay and ac...
2,residual_return_vs_universe_20_h20,residual_return_vs_universe_20,20,cross_sectional_relative_value,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK,WATCHLIST_DIVERSIFIER,54.0,NaN,NaN,...,HIGH_DRAWDOWN,0.046677,0,NaN,NaN,NaN,NaN,0.264721,1,Watchlist diversifier with stable decay and ac...
3,vol_of_vol_20_h20,vol_of_vol_20,20,volatility_structure,POSITIVE_EDGE,WEAK,WATCHLIST_DIVERSIFIER,54.0,NaN,NaN,...,DOWNTREND,0.079858,0,NaN,NaN,NaN,NaN,0.153123,1,Watchlist diversifier with stable decay and ac...


## 5. Load candidate signal panels


In [4]:
ORTHOGONAL_V2_CANDIDATE_SPECS = [
    ('vol_surprise_20_60', 20),
    ('price_impact_proxy_20', 20),
    ('range_expansion_failure_5', 20),
    ('liquidity_adjusted_reversal_5', 5),
]
orthogonal_v2_signal_names = [signal_name for signal_name, _ in ORTHOGONAL_V2_CANDIDATE_SPECS]
needed_signals = sorted(
    set(approved_input_signals['signal_name'].dropna().unique().tolist())
    | set(watchlist_diversifier_signals['signal_name'].dropna().unique().tolist())
    | set(alpha_signal_pool['signal_name'].dropna().unique().tolist())
    | set(orthogonal_v2_signal_names)
)
load_start = time.perf_counter()
candidate_signals_filtered = load_candidate_signals_for_names(needed_signals, db_path=sqlite_db_path)
candidate_signal_load_seconds = time.perf_counter() - load_start
inputs['candidate_signals'] = candidate_signals_filtered

candidate_signal_run_ids = sorted(candidate_signals_filtered.get('run_id', pd.Series(dtype='object')).dropna().astype(str).unique().tolist())
candidate_signal_versions = sorted(candidate_signals_filtered.get('signal_version', pd.Series(dtype='object')).dropna().astype(str).unique().tolist())
clean_price_tickers = [column for column in inputs['close_prices'].columns if column != 'SPY']
filtered_signal_tickers = sorted(candidate_signals_filtered['ticker'].dropna().astype(str).unique().tolist()) if 'ticker' in candidate_signals_filtered.columns else []
universe_metadata = load_table('universe_metadata_current', db_path=sqlite_db_path) if table_exists('universe_metadata_current', db_path=sqlite_db_path) else pd.DataFrame()
universe_versions = sorted(universe_metadata.get('universe_version', pd.Series(dtype='object')).dropna().astype(str).unique().tolist())
universe_names = sorted(universe_metadata.get('universe_name', pd.Series(dtype='object')).dropna().astype(str).unique().tolist())
universe_run_ids = sorted(universe_metadata.get('run_id', pd.Series(dtype='object')).dropna().astype(str).unique().tolist())

input_shapes = pd.DataFrame(
    [
        {'input_name': 'candidate_signals_filtered', 'rows': len(candidate_signals_filtered), 'columns': len(candidate_signals_filtered.columns)},
        {'input_name': 'regime_features_ic_current', 'rows': len(inputs['regime_features']), 'columns': len(inputs['regime_features'].columns)},
        {'input_name': 'alpha_signal_pool', 'rows': len(alpha_signal_pool), 'columns': len(alpha_signal_pool.columns)},
        {'input_name': 'clean_close_prices_current', 'rows': len(inputs['close_prices']), 'columns': len(inputs['close_prices'].columns)},
    ]
)
lineage_diagnostics = pd.DataFrame(
    [
        {'lineage_item': 'candidate_signals_filtered_run_id', 'value': ', '.join(candidate_signal_run_ids)},
        {'lineage_item': 'candidate_signals_filtered_signal_version', 'value': ', '.join(candidate_signal_versions)},
        {'lineage_item': 'candidate_signals_filtered_n_tickers', 'value': len(filtered_signal_tickers)},
        {'lineage_item': 'candidate_signals_filtered_rows_loaded', 'value': len(candidate_signals_filtered)},
        {'lineage_item': 'candidate_signals_filtered_load_seconds', 'value': round(candidate_signal_load_seconds, 3)},
        {'lineage_item': 'clean_close_prices_shape', 'value': str(inputs['close_prices'].shape)},
        {'lineage_item': 'clean_close_prices_n_tickers_ex_spy', 'value': len(clean_price_tickers)},
        {'lineage_item': 'universe_metadata_run_id', 'value': ', '.join(universe_run_ids)},
        {'lineage_item': 'universe_name', 'value': ', '.join(universe_names)},
        {'lineage_item': 'universe_version', 'value': ', '.join(universe_versions)},
        {'lineage_item': 'needed_signal_count', 'value': len(needed_signals)},
        {'lineage_item': 'smoothing_window', 'value': SMOOTHING_WINDOW},
        {'lineage_item': 'rebalance_frequency', 'value': REBALANCE_FREQUENCY},
        {'lineage_item': 'turnover_control_enabled', 'value': TURNOVER_CONTROL_ENABLED},
        {'lineage_item': 'turnover_control_update_rate', 'value': TURNOVER_CONTROL_UPDATE_RATE},
    ]
)

print(f'Loaded {len(candidate_signals_filtered):,} candidate signal rows for {len(needed_signals)} needed signals in {candidate_signal_load_seconds:.2f}s')
print('04A input shapes')
display(input_shapes)
print('04A expanded-universe lineage diagnostics')
display(lineage_diagnostics)


Loaded 6,017,064 candidate signal rows for 6 needed signals in 10.30s
04A input shapes


,input_name,rows,columns
0,candidate_signals_filtered,6017064,6
1,regime_features_ic_current,2098,11
2,alpha_signal_pool,4,27
3,clean_close_prices_current,2098,478


04A expanded-universe lineage diagnostics


,lineage_item,value
0,candidate_signals_filtered_run_id,phase2_orthogonal_signals_20260508_075857
1,candidate_signals_filtered_signal_version,
2,candidate_signals_filtered_n_tickers,478
3,candidate_signals_filtered_rows_loaded,6017064
4,candidate_signals_filtered_load_seconds,10.299
5,clean_close_prices_shape,"(2098, 478)"
6,clean_close_prices_n_tickers_ex_spy,477
7,universe_metadata_run_id,phase2_nb01_20260507_223029
8,universe_name,"dynamic_top300_from_current_large_liquid_pool,..."
9,universe_version,dynamic_top300_from_current_large_liquid_pool_v1


## 6. Build normalized signal panels


In [5]:
normalized_signal_panels = build_normalized_signal_panels(
    approved_signals=approved_input_signals,
    candidate_signals=candidate_signals_filtered,
)
normalized_panel_summary = pd.DataFrame(
    [
        {
            'component_id': component_id,
            'n_dates': panel.shape[0],
            'n_tickers': panel.shape[1],
            'finite_pct': panel.notna().to_numpy().mean(),
        }
        for component_id, panel in normalized_signal_panels.items()
    ]
)
display(normalized_panel_summary)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after building normalized signal panels')


""


## 7. Build equal-weight alpha


In [6]:
eligibility_membership = load_table('universe_membership_dynamic_top300_current', db_path=sqlite_db_path)
eligible_mask, warmup_cutoff = build_dynamic_universe_eligibility_mask(
    membership=eligibility_membership,
    close_prices=inputs['close_prices'],
    warmup_trading_days=WARMUP_TRADING_DAYS,
)
eligible_mask_for_construction = eligible_mask.reindex(
    index=inputs['close_prices'].index,
    columns=inputs['close_prices'].columns,
    fill_value=False,
).astype(bool)

orthogonal_v2_daily_ic = (
    load_table('signal_regime_ic_daily_current', db_path=sqlite_db_path)
    if table_exists('signal_regime_ic_daily_current', db_path=sqlite_db_path)
    else pd.DataFrame()
)
orthogonal_v2_health = (
    load_table('signal_health_score_current', db_path=sqlite_db_path)
    if table_exists('signal_health_score_current', db_path=sqlite_db_path)
    else pd.DataFrame()
)
orthogonal_v2_component_metrics = pd.DataFrame(
    ORTHOGONAL_V2_CANDIDATE_SPECS,
    columns=['signal_name', 'horizon'],
)
if not orthogonal_v2_health.empty:
    health_columns = [
        'signal_name',
        'horizon',
        'signal_family',
        'signal_direction',
        'signal_health_score',
        'signal_health_gate',
    ]
    orthogonal_v2_component_metrics = orthogonal_v2_component_metrics.merge(
        orthogonal_v2_health[[column for column in health_columns if column in orthogonal_v2_health.columns]],
        on=['signal_name', 'horizon'],
        how='left',
    )
if not orthogonal_v2_daily_ic.empty:
    ic = orthogonal_v2_daily_ic.loc[
        orthogonal_v2_daily_ic.get('regime_column', pd.Series(dtype='object')).astype(str).eq('benchmark_vol_regime')
    ].copy()
    ic['Date'] = pd.to_datetime(ic['Date'], errors='coerce')
    ic['horizon'] = pd.to_numeric(ic['horizon'], errors='coerce')
    ic['daily_ic'] = pd.to_numeric(ic['daily_ic'], errors='coerce')
    unique_dates = np.array(sorted(ic['Date'].dropna().unique()))
    date_windows = {date: idx + 1 for idx, dates in enumerate(np.array_split(unique_dates, 4)) for date in dates}
    ic['window_id'] = ic['Date'].map(date_windows)
    ic_summary = (
        ic.groupby(['signal_name', 'horizon'])
        .agg(
            mean_ic=('daily_ic', 'mean'),
            ic_std=('daily_ic', 'std'),
            sign_consistency=('daily_ic', lambda values: float((values > 0).mean())),
        )
        .reset_index()
    )
    ic_summary['ic_ir'] = ic_summary['mean_ic'] / ic_summary['ic_std'].replace(0.0, np.nan)
    ic_windows = (
        ic.groupby(['signal_name', 'horizon', 'window_id'])
        .agg(window_mean_ic=('daily_ic', 'mean'))
        .reset_index()
    )
    ic_windows['positive_window'] = ic_windows['window_mean_ic'].gt(0)
    persistence = (
        ic_windows.groupby(['signal_name', 'horizon'])
        .agg(persistence_ratio=('positive_window', 'mean'))
        .reset_index()
    )
    orthogonal_v2_component_metrics = (
        orthogonal_v2_component_metrics
        .merge(ic_summary, on=['signal_name', 'horizon'], how='left')
        .merge(persistence, on=['signal_name', 'horizon'], how='left')
    )

orthogonal_v2_component_metrics[['mean_ic', 'persistence_ratio', 'sign_consistency']] = orthogonal_v2_component_metrics[[
    'mean_ic',
    'persistence_ratio',
    'sign_consistency',
]].fillna({'mean_ic': 0.0, 'persistence_ratio': 0.50, 'sign_consistency': 0.50})
orthogonal_v2_component_metrics['signal_direction'] = orthogonal_v2_component_metrics.get(
    'signal_direction',
    pd.Series('POSITIVE_EDGE', index=orthogonal_v2_component_metrics.index),
).fillna('POSITIVE_EDGE')

print('Orthogonal v2 rule-weight component metrics')
display(orthogonal_v2_component_metrics)

alpha_candidates, _, alpha_dynamic_weight_audit, dynamic_component_stats = build_alpha_candidates(
    approved_signals=approved_input_signals,
    candidate_signals=candidate_signals_filtered,
    regime_features=inputs['regime_features'],
    watchlist_diversifiers=watchlist_diversifier_signals,
    signal_pool=alpha_signal_pool,
    close_prices=inputs['close_prices'],
    eligible_mask=eligible_mask_for_construction,
    orthogonal_v2_component_metrics=orthogonal_v2_component_metrics,
    smoothing_window=SMOOTHING_WINDOW,
    rebalance_frequency=REBALANCE_FREQUENCY,
    update_rate=TURNOVER_CONTROL_UPDATE_RATE,
    turnover_control_enabled=TURNOVER_CONTROL_ENABLED,
)

if 'alpha_equal_weight_research_v1' in alpha_candidates:
    equal_weight_alpha = alpha_candidates['alpha_equal_weight_research_v1']
    display(equal_weight_alpha.tail())
else:
    equal_weight_alpha = pd.DataFrame()
    print('alpha_equal_weight_research_v1 was not constructed because no approved core research components were available.')


Orthogonal v2 rule-weight component metrics


,signal_name,horizon,signal_family,signal_direction,signal_health_score,signal_health_gate,mean_ic,ic_std,sign_consistency,ic_ir,persistence_ratio
0,vol_surprise_20_60,20,volatility_structure,POSITIVE_EDGE,0.0,REJECTED_RESEARCH,0.007711,0.135547,0.493327,0.056888,0.75
1,price_impact_proxy_20,20,liquidity_flow,POSITIVE_EDGE,27.0,REJECTED_RESEARCH,0.011528,0.133481,0.502860,0.086367,0.75
2,range_expansion_failure_5,20,volatility_structure,POSITIVE_EDGE,58.0,WATCHLIST_RESEARCH,0.014333,0.127624,0.527169,0.112310,1.00
3,liquidity_adjusted_reversal_5,5,liquidity_flow,POSITIVE_EDGE,5.0,REJECTED_RESEARCH,0.007327,0.166707,0.504766,0.043952,1.00


alpha_equal_weight_research_v1 was not constructed because no approved core research components were available.


## 8. Build health-weighted alpha


In [7]:
if 'alpha_health_weighted_research_v1' in alpha_candidates:
    health_weighted_alpha = alpha_candidates['alpha_health_weighted_research_v1']
    display(health_weighted_alpha.tail())
else:
    health_weighted_alpha = pd.DataFrame()
    print('alpha_health_weighted_research_v1 was not constructed because no approved core research components were available.')


alpha_health_weighted_research_v1 was not constructed because no approved core research components were available.


## 9. Build regime-aware alpha


In [8]:
if 'alpha_regime_aware_research_v1' in alpha_candidates:
    regime_aware_alpha = alpha_candidates['alpha_regime_aware_research_v1']
    display(regime_aware_alpha.tail())
else:
    regime_aware_alpha = pd.DataFrame()
    print('alpha_regime_aware_research_v1 was not constructed because no approved core research components were available.')


alpha_regime_aware_research_v1 was not constructed because no approved core research components were available.


## 10. Build metadata and quality report


In [9]:
reference_panel = next(iter(alpha_candidates.values())) if alpha_candidates else pd.DataFrame()
eligible_mask_for_alphas = eligible_mask.reindex(
    index=reference_panel.index,
    columns=reference_panel.columns,
    fill_value=False,
).astype(bool) if not reference_panel.empty else eligible_mask
panel_cells = int(reference_panel.shape[0] * reference_panel.shape[1]) if not reference_panel.empty else 0
eligible_cells = int(eligible_mask_for_alphas.to_numpy(dtype=bool).sum()) if not eligible_mask_for_alphas.empty else 0
eligibility_diagnostics = pd.DataFrame([
    {'metric': 'total_panel_cells', 'value': panel_cells},
    {'metric': 'eligible_mask_cells', 'value': eligible_cells},
    {'metric': 'eligible_denominator_reduction_pct', 'value': 1.0 - (eligible_cells / panel_cells if panel_cells else 0.0)},
    {'metric': 'warmup_cutoff', 'value': warmup_cutoff},
    {'metric': 'warmup_trading_days', 'value': WARMUP_TRADING_DAYS},
    {'metric': 'eligible_tickers_with_at_least_one_day', 'value': int(eligible_mask_for_alphas.any(axis=0).sum()) if not eligible_mask_for_alphas.empty else 0},
])

alpha_construction_metadata = build_alpha_construction_metadata(
    alpha_candidates=alpha_candidates,
    approved_signals=approved_input_signals,
    run_id=run_id,
    alpha_construction_version=ALPHA_CONSTRUCTION_VERSION,
    watchlist_diversifiers=watchlist_diversifier_signals,
    signal_pool=alpha_signal_pool,
    smoothing_window=SMOOTHING_WINDOW,
    rebalance_frequency=REBALANCE_FREQUENCY,
    update_rate=TURNOVER_CONTROL_UPDATE_RATE,
    turnover_control_enabled=TURNOVER_CONTROL_ENABLED,
    dynamic_component_stats=dynamic_component_stats,
)
alpha_construction_quality = build_alpha_construction_quality(
    alpha_candidates=alpha_candidates,
    run_id=run_id,
    alpha_construction_version=ALPHA_CONSTRUCTION_VERSION,
    eligible_mask=eligible_mask_for_alphas,
)
alpha_construction_diagnostics = build_alpha_construction_diagnostics(
    alpha_panels=alpha_candidates,
    run_id=run_id,
    alpha_construction_version=ALPHA_CONSTRUCTION_VERSION,
    dynamic_component_stats=dynamic_component_stats,
    eligible_mask=eligible_mask_for_alphas,
)
alpha_construction_correlation = build_alpha_correlation_matrix(
    alpha_panels=alpha_candidates,
    run_id=run_id,
    alpha_construction_version=ALPHA_CONSTRUCTION_VERSION,
)

orthogonal_alpha_name = 'alpha_orthogonal_diversifier_v1_smooth'
v4_alpha_names = [
    'alpha_decay_aware_dynamic_v4_smooth',
    'alpha_rolling_ic_dynamic_v4_smooth',
    'alpha_regime_blend_dynamic_v4_smooth',
    'alpha_hybrid_adaptive_v4_smooth',
]
sleeve_correlation_report = pd.DataFrame()
if not alpha_construction_correlation.empty and orthogonal_alpha_name in alpha_candidates:
    sleeve_correlation_report = alpha_construction_correlation.loc[
        (alpha_construction_correlation['alpha_name_1'].eq(orthogonal_alpha_name) & alpha_construction_correlation['alpha_name_2'].isin(v4_alpha_names))
        | (alpha_construction_correlation['alpha_name_2'].eq(orthogonal_alpha_name) & alpha_construction_correlation['alpha_name_1'].isin(v4_alpha_names))
    ].copy()
    if not sleeve_correlation_report.empty:
        sleeve_correlation_report['comparison_alpha'] = np.where(
            sleeve_correlation_report['alpha_name_1'].eq(orthogonal_alpha_name),
            sleeve_correlation_report['alpha_name_2'],
            sleeve_correlation_report['alpha_name_1'],
        )
        sleeve_correlation_report['abs_correlation'] = pd.to_numeric(
            sleeve_correlation_report['correlation'],
            errors='coerce',
        ).abs()
        sleeve_correlation_report['orthogonality_flag'] = np.where(
            sleeve_correlation_report['abs_correlation'].gt(0.70),
            'ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70',
            'ORTHOGONALITY_PRESERVED_ABS_CORR_LE_0P70',
        )
        sleeve_correlation_report = sleeve_correlation_report[[
            'alpha_name_1',
            'alpha_name_2',
            'comparison_alpha',
            'correlation',
            'abs_correlation',
            'orthogonality_flag',
        ]].sort_values('abs_correlation', ascending=False)
quality_status_counts = alpha_construction_quality['status'].value_counts()
alpha_candidate_count = len(alpha_candidates)

quality_diagnostic_display = (
    alpha_construction_quality[[
        'alpha_name', 'finite_pct', 'missing_pct', 'avg_turnover_proxy', 'max_abs_alpha', 'status', 'quality_notes'
    ]]
    .merge(
        alpha_construction_diagnostics[[
            'alpha_name', 'median_turnover_proxy', 'max_turnover_proxy'
        ]],
        on='alpha_name',
        how='left',
    )
    .sort_values('alpha_name')
)
old_quality_rows = []
post_warmup_mask = pd.DataFrame(
    True,
    index=eligible_mask_for_alphas.index,
    columns=eligible_mask_for_alphas.columns,
) if not eligible_mask_for_alphas.empty else pd.DataFrame()
if not post_warmup_mask.empty:
    post_warmup_mask = post_warmup_mask & pd.DataFrame(
        np.repeat(
            pd.Series(post_warmup_mask.index >= warmup_cutoff, index=post_warmup_mask.index).to_numpy()[:, None],
            len(post_warmup_mask.columns),
            axis=1,
        ),
        index=post_warmup_mask.index,
        columns=post_warmup_mask.columns,
    )
for alpha_name, panel in alpha_candidates.items():
    values = panel.to_numpy(dtype=float)
    finite = pd.DataFrame(pd.notna(panel).to_numpy(), index=panel.index, columns=panel.columns)
    old_total = int(values.size)
    old_finite = int(finite.to_numpy(dtype=bool).sum())
    eligible_aligned = eligible_mask_for_alphas.reindex(index=panel.index, columns=panel.columns, fill_value=False).astype(bool)
    eligible_total = int(eligible_aligned.to_numpy(dtype=bool).sum())
    eligible_finite = int((finite & eligible_aligned).to_numpy(dtype=bool).sum())
    warmup_aligned = post_warmup_mask.reindex(index=panel.index, columns=panel.columns, fill_value=False).astype(bool) if not post_warmup_mask.empty else eligible_aligned
    warmup_total = int(warmup_aligned.to_numpy(dtype=bool).sum())
    warmup_finite = int((finite & warmup_aligned).to_numpy(dtype=bool).sum())
    old_quality_rows.append({
        'alpha_name': alpha_name,
        'total_panel_cells': old_total,
        'eligible_mask_cells': eligible_total,
        'denominator_reduction_pct': 1.0 - (eligible_total / old_total if old_total else 0.0),
        'finite_pct_old_full_panel': old_finite / old_total if old_total else 0.0,
        'finite_pct_excluding_first_60_days': warmup_finite / warmup_total if warmup_total else 0.0,
        'finite_pct_eligible_only': eligible_finite / eligible_total if eligible_total else 0.0,
    })
coverage_comparison_display = pd.DataFrame(old_quality_rows).sort_values('alpha_name')

v4_pairs = [
    ('alpha_decay_aware_dynamic_v3', 'alpha_decay_aware_dynamic_v4_smooth'),
    ('alpha_rolling_ic_dynamic_v3', 'alpha_rolling_ic_dynamic_v4_smooth'),
    ('alpha_regime_blend_dynamic_v3', 'alpha_regime_blend_dynamic_v4_smooth'),
    ('alpha_hybrid_adaptive_v3', 'alpha_hybrid_adaptive_v4_smooth'),
]
quality_lookup = alpha_construction_quality.set_index('alpha_name')
comparison_rows = []
for raw_name, smooth_name in v4_pairs:
    if raw_name in quality_lookup.index and smooth_name in quality_lookup.index:
        comparison_rows.append({
            'alpha_name_raw_v3': raw_name,
            'alpha_name_v4': smooth_name,
            'finite_pct_delta': quality_lookup.at[smooth_name, 'finite_pct'] - quality_lookup.at[raw_name, 'finite_pct'],
            'avg_turnover_proxy_delta': quality_lookup.at[smooth_name, 'avg_turnover_proxy'] - quality_lookup.at[raw_name, 'avg_turnover_proxy'],
            'quality_status_raw': quality_lookup.at[raw_name, 'status'],
            'quality_status_v4': quality_lookup.at[smooth_name, 'status'],
            'quality_notes_v4': quality_lookup.at[smooth_name, 'quality_notes'],
        })
v3_v4_comparison = pd.DataFrame(comparison_rows)

print('Eligibility mask diagnostics')
display(eligibility_diagnostics)
print('Coverage old/full-panel vs eligible-mask diagnostics')
display(coverage_comparison_display)
display(alpha_construction_metadata)
display(alpha_construction_quality)
display(alpha_construction_diagnostics)
orthogonal_v2_component_score_table = pd.DataFrame()
orthogonal_v2_score_json = dynamic_component_stats.get(
    'alpha_orthogonal_diversifier_v2_score_weighted_smooth__component_scores',
    {},
).get('score_table')
if orthogonal_v2_score_json:
    orthogonal_v2_component_score_table = pd.read_json(orthogonal_v2_score_json)

print('Orthogonal v2 component score table')
display(orthogonal_v2_component_score_table)
print('v3/v4 quality diagnostics')
display(quality_diagnostic_display)
print('v3 raw vs v4 smooth comparison')
display(v3_v4_comparison)
print('Orthogonal sleeve correlation report')
display(sleeve_correlation_report)
display(alpha_construction_correlation)


/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/src/alpha_construction.py:1508: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  v2_score_table = pd.read_json(v2_score_json)


Eligibility mask diagnostics


,metric,value
0,total_panel_cells,1002844
1,eligible_mask_cells,611399
2,eligible_denominator_reduction_pct,0.390335
3,warmup_cutoff,2018-03-29 00:00:00
4,warmup_trading_days,60
5,eligible_tickers_with_at_least_one_day,461


Coverage old/full-panel vs eligible-mask diagnostics


,alpha_name,total_panel_cells,eligible_mask_cells,denominator_reduction_pct,finite_pct_old_full_panel,finite_pct_excluding_first_60_days,finite_pct_eligible_only
0,alpha_decay_aware_dynamic_v3,1002844,611399,0.390335,0.889148,0.910440,0.995785
4,alpha_decay_aware_dynamic_v4_smooth,1002844,611399,0.390335,0.888534,0.910146,0.995625
3,alpha_hybrid_adaptive_v3,1002844,611399,0.390335,0.885011,0.906181,0.995376
7,alpha_hybrid_adaptive_v4_smooth,1002844,611399,0.390335,0.888310,0.909915,0.995486
8,alpha_orthogonal_diversifier_v1_smooth,1002844,611399,0.390335,0.880282,0.906199,0.991994
9,alpha_orthogonal_diversifier_v2_score_weighted...,1002844,611399,0.390335,0.887676,0.909263,0.995255
1,alpha_regime_blend_dynamic_v3,1002844,611399,0.390335,0.889148,0.910440,0.995785
5,alpha_regime_blend_dynamic_v4_smooth,1002844,611399,0.390335,0.888534,0.910146,0.995625
2,alpha_rolling_ic_dynamic_v3,1002844,611399,0.390335,0.885011,0.906181,0.995376
6,alpha_rolling_ic_dynamic_v4_smooth,1002844,611399,0.390335,0.888310,0.909915,0.995486


...

v3 raw vs v4 smooth comparison


,alpha_name_raw_v3,alpha_name_v4,finite_pct_delta,avg_turnover_proxy_delta,quality_status_raw,quality_status_v4,quality_notes_v4
0,alpha_decay_aware_dynamic_v3,alpha_decay_aware_dynamic_v4_smooth,-0.00016,-8.609327,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno..."
1,alpha_rolling_ic_dynamic_v3,alpha_rolling_ic_dynamic_v4_smooth,0.00011,-8.122439,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno..."
2,alpha_regime_blend_dynamic_v3,alpha_regime_blend_dynamic_v4_smooth,-0.00016,-8.812789,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno..."
3,alpha_hybrid_adaptive_v3,alpha_hybrid_adaptive_v4_smooth,0.00011,-8.548351,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno..."


Orthogonal sleeve correlation report


,alpha_name_1,alpha_name_2,comparison_alpha,correlation,abs_correlation,orthogonality_flag
48,alpha_decay_aware_dynamic_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,alpha_decay_aware_dynamic_v4_smooth,0.834848,0.834848,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
84,alpha_orthogonal_diversifier_v1_smooth,alpha_decay_aware_dynamic_v4_smooth,alpha_decay_aware_dynamic_v4_smooth,0.834848,0.834848,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
78,alpha_hybrid_adaptive_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,alpha_hybrid_adaptive_v4_smooth,0.830409,0.830409,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
87,alpha_orthogonal_diversifier_v1_smooth,alpha_hybrid_adaptive_v4_smooth,alpha_hybrid_adaptive_v4_smooth,0.830409,0.830409,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
58,alpha_regime_blend_dynamic_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,alpha_regime_blend_dynamic_v4_smooth,0.820274,0.820274,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
85,alpha_orthogonal_diversifier_v1_smooth,alpha_regime_blend_dynamic_v4_smooth,alpha_regime_blend_dynamic_v4_smooth,0.820274,0.820274,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
68,alpha_rolling_ic_dynamic_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,alpha_rolling_ic_dynamic_v4_smooth,0.747806,0.747806,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
86,alpha_orthogonal_diversifier_v1_smooth,alpha_rolling_ic_dynamic_v4_smooth,alpha_rolling_ic_dynamic_v4_smooth,0.747806,0.747806,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70


,alpha_name_1,alpha_name_2,correlation,run_id,alpha_construction_version
0,alpha_decay_aware_dynamic_v3,alpha_decay_aware_dynamic_v3,1.000000,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
1,alpha_decay_aware_dynamic_v3,alpha_regime_blend_dynamic_v3,0.995534,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
2,alpha_decay_aware_dynamic_v3,alpha_rolling_ic_dynamic_v3,0.874741,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
3,alpha_decay_aware_dynamic_v3,alpha_hybrid_adaptive_v3,0.992280,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
4,alpha_decay_aware_dynamic_v3,alpha_decay_aware_dynamic_v4_smooth,0.318224,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
...,...,...,...,...,...
95,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_regime_blend_dynamic_v4_smooth,0.308254,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
96,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_rolling_ic_dynamic_v4_smooth,0.276471,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
97,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_hybrid_adaptive_v4_smooth,0.295364,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
98,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_orthogonal_diversifier_v1_smooth,0.052754,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4


## 11. Save outputs to SQLite


In [10]:
saved_paths = save_alpha_construction_outputs(
    alpha_candidates=alpha_candidates,
    metadata=alpha_construction_metadata,
    quality=alpha_construction_quality,
    diagnostics=alpha_construction_diagnostics,
    correlation=alpha_construction_correlation,
    signal_pool=alpha_signal_pool,
    dynamic_weight_audit=alpha_dynamic_weight_audit,
    db_path=sqlite_db_path,
    run_id=run_id,
    alpha_construction_version=ALPHA_CONSTRUCTION_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in ALPHA_CONSTRUCTION_TABLES.items()
    ]
)

display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving alpha construction outputs')


,artifact,current_table,history_table,sqlite_path
0,candidates,alpha_constructed_candidates_current,alpha_constructed_candidates_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,alpha_construction_metadata_current,alpha_construction_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,alpha_construction_quality_current,alpha_construction_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,diagnostics,alpha_construction_diagnostics_current,alpha_construction_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,correlation,alpha_construction_correlation_current,alpha_construction_correlation_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,signal_pool,alpha_signal_pool_current,alpha_signal_pool_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,dynamic_weight_audit,alpha_dynamic_weight_audit_current,alpha_dynamic_weight_audit_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 12. Final display


In [11]:
print('Approved/research input signals')
display(approved_input_signals)

print('Approved/research input signal pool')
display(alpha_signal_pool)

print('Watchlist diversifier signals selected')
display(pool_watchlist_diversifiers)

print('Signal pool summary by source_role and family')
display(
    alpha_signal_pool.groupby(['source_role', 'signal_family'], dropna=False)
    .agg(n_components=('component_id', 'nunique'), avg_pool_weight=('pool_weight_base', 'mean'))
    .reset_index()
    .sort_values(['source_role', 'n_components'], ascending=[True, False])
)


alpha_candidate_panel_summary = pd.DataFrame(
    [
        {
            'alpha_name': alpha_name,
            'n_dates': panel.shape[0],
            'n_tickers': panel.shape[1],
            'first_date': panel.index.min(),
            'last_date': panel.index.max(),
        }
        for alpha_name, panel in alpha_candidates.items()
    ]
)
print('Constructed alpha panel ticker counts after 04A run')
display(alpha_candidate_panel_summary.sort_values('alpha_name'))

print('Eligibility mask diagnostics')
display(eligibility_diagnostics)

print('Coverage old/full-panel vs eligible-mask diagnostics')
display(coverage_comparison_display)

print('Alpha quality table')
display(alpha_construction_quality)

print('v3/v4 quality diagnostics')
display(quality_diagnostic_display)

print('v3 raw vs v4 smooth comparison')
display(v3_v4_comparison)

print('Orthogonal sleeve correlation report')
display(sleeve_correlation_report)

print('Diagnostics table')
display(alpha_construction_diagnostics)

print('Dynamic weight audit sample')
display(alpha_dynamic_weight_audit.head(50))

print('Alpha correlation matrix')
display(alpha_construction_correlation)

print('SQLite tables written')
display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving alpha construction outputs')


Approved/research input signals


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version


Approved/research input signal pool


,component_id,signal_name,horizon,signal_family,signal_direction,signal_strength,source_role,signal_health_score,final_research_gate,reproducibility_status,...,best_regime_value,adjusted_best_abs_ic,selected_flag,diversity_group,diversity_candidate_tier,signal_source,orthogonal_version,pool_weight_base,pool_eligible_flag,pool_reason
0,vol_of_vol_20_h10,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,DIVERSITY_SELECTED,62.0,WATCHLIST_ALPHA_RESEARCH,CONDITIONAL_PASS,...,DOWNTREND,0.062676,1,ORTHOGONAL_DIVERSIFIER_SELECTED,ORTHOGONAL_DIVERSIFIER,orthogonal_generated,phase2_orthogonal_signals_v2,0.294920,1,Selected by 03G diversity engine.
1,range_expansion_failure_5_h20,range_expansion_failure_5,20,volatility_structure,POSITIVE_EDGE,WEAK,WATCHLIST_DIVERSIFIER,58.0,NaN,NaN,...,HIGH_VOL,0.031891,0,NaN,NaN,NaN,NaN,0.287236,1,Watchlist diversifier with stable decay and ac...
2,residual_return_vs_universe_20_h20,residual_return_vs_universe_20,20,cross_sectional_relative_value,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK,WATCHLIST_DIVERSIFIER,54.0,NaN,NaN,...,HIGH_DRAWDOWN,0.046677,0,NaN,NaN,NaN,NaN,0.264721,1,Watchlist diversifier with stable decay and ac...
3,vol_of_vol_20_h20,vol_of_vol_20,20,volatility_structure,POSITIVE_EDGE,WEAK,WATCHLIST_DIVERSIFIER,54.0,NaN,NaN,...,DOWNTREND,0.079858,0,NaN,NaN,NaN,NaN,0.153123,1,Watchlist diversifier with stable decay and ac...


Watchlist diversifier signals selected


...

,alpha_name,Date,signal_name,horizon,component_id,weight,weight_method,regime_match_flag,rolling_ic_used
0,alpha_decay_aware_dynamic_v3,2018-01-02,vol_of_vol_20,10,vol_of_vol_20_h10,0.294920,decay_aware_base,NaN,NaN
1,alpha_decay_aware_dynamic_v3,2018-01-02,range_expansion_failure_5,20,range_expansion_failure_5_h20,0.287236,decay_aware_base,NaN,NaN
2,alpha_decay_aware_dynamic_v3,2018-01-02,residual_return_vs_universe_20,20,residual_return_vs_universe_20_h20,0.264721,decay_aware_base,NaN,NaN
3,alpha_decay_aware_dynamic_v3,2018-01-02,vol_of_vol_20,20,vol_of_vol_20_h20,0.153123,decay_aware_base,NaN,NaN
4,alpha_decay_aware_dynamic_v3,2018-01-03,vol_of_vol_20,10,vol_of_vol_20_h10,0.294920,decay_aware_base,NaN,NaN
5,alpha_decay_aware_dynamic_v3,2018-01-03,range_expansion_failure_5,20,range_expansion_failure_5_h20,0.287236,decay_aware_base,NaN,NaN
6,alpha_decay_aware_dynamic_v3,2018-01-03,residual_return_vs_universe_20,20,residual_return_vs_universe_20_h20,0.264721,decay_aware_base,NaN,NaN
7,alpha_decay_aware_dynamic_v3,2018-01-03,vol_of_vol_20,20,vol_of_vol_20_h20,0.153123,decay_aware_base,NaN,NaN
8,alpha_decay_aware_dynamic_v3,2018-01-04,vol_of_vol_20,10,vol_of_vol_20_h10,0.294920,decay_aware_base,NaN,NaN
9,alpha_decay_aware_dynamic_v3,2018-01-04,range_expansion_failure_5,20,range_expansion_failure_5_h20,0.287236,decay_aware_base,NaN,NaN


Alpha correlation matrix


,alpha_name_1,alpha_name_2,correlation,run_id,alpha_construction_version
0,alpha_decay_aware_dynamic_v3,alpha_decay_aware_dynamic_v3,1.000000,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
1,alpha_decay_aware_dynamic_v3,alpha_regime_blend_dynamic_v3,0.995534,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
2,alpha_decay_aware_dynamic_v3,alpha_rolling_ic_dynamic_v3,0.874741,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
3,alpha_decay_aware_dynamic_v3,alpha_hybrid_adaptive_v3,0.992280,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
4,alpha_decay_aware_dynamic_v3,alpha_decay_aware_dynamic_v4_smooth,0.318224,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
...,...,...,...,...,...
95,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_regime_blend_dynamic_v4_smooth,0.308254,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
96,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_rolling_ic_dynamic_v4_smooth,0.276471,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
97,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_hybrid_adaptive_v4_smooth,0.295364,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
98,alpha_orthogonal_diversifier_v2_score_weighted...,alpha_orthogonal_diversifier_v1_smooth,0.052754,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,candidates,alpha_constructed_candidates_current,alpha_constructed_candidates_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,alpha_construction_metadata_current,alpha_construction_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,alpha_construction_quality_current,alpha_construction_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,diagnostics,alpha_construction_diagnostics_current,alpha_construction_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,correlation,alpha_construction_correlation_current,alpha_construction_correlation_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,signal_pool,alpha_signal_pool_current,alpha_signal_pool_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,dynamic_weight_audit,alpha_dynamic_weight_audit_current,alpha_dynamic_weight_audit_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [12]:
from src.db import load_table

quality = load_table("alpha_construction_quality_current")
diag = load_table("alpha_construction_diagnostics_current")
pool = load_table("alpha_signal_pool_current")
weights = load_table("alpha_dynamic_weight_audit_current")
corr = load_table("alpha_construction_correlation_current")

display(pool["source_role"].value_counts())
display(pool["signal_family"].value_counts())

display(quality.sort_values("alpha_name"))
display(diag.sort_values("alpha_name"))

display(
    weights.groupby("alpha_name")["weight"]
    .agg(["count", "mean", "min", "max"])
)

display(corr.sort_values("correlation"))

constructed = load_table("alpha_constructed_candidates_current")
constructed_current_ticker_counts = (
    constructed.groupby("alpha_name")["ticker"]
    .nunique()
    .rename("n_tickers")
    .reset_index()
    .sort_values("alpha_name")
)
print("alpha_constructed_candidates_current ticker counts")
display(constructed_current_ticker_counts)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after alpha construction diagnostic readback')


current_quality_diag = (
    quality[[
        'alpha_name', 'finite_pct', 'missing_pct', 'avg_turnover_proxy', 'max_abs_alpha', 'status', 'quality_notes'
    ]]
    .merge(
        diag[['alpha_name', 'median_turnover_proxy', 'max_turnover_proxy']],
        on='alpha_name',
        how='left',
    )
    .sort_values('alpha_name')
)
current_pairs = [
    ('alpha_decay_aware_dynamic_v3', 'alpha_decay_aware_dynamic_v4_smooth'),
    ('alpha_rolling_ic_dynamic_v3', 'alpha_rolling_ic_dynamic_v4_smooth'),
    ('alpha_regime_blend_dynamic_v3', 'alpha_regime_blend_dynamic_v4_smooth'),
    ('alpha_hybrid_adaptive_v3', 'alpha_hybrid_adaptive_v4_smooth'),
]
current_lookup = quality.set_index('alpha_name')
current_comparison_rows = []
for raw_name, smooth_name in current_pairs:
    if raw_name in current_lookup.index and smooth_name in current_lookup.index:
        current_comparison_rows.append({
            'alpha_name_raw_v3': raw_name,
            'alpha_name_v4': smooth_name,
            'finite_pct_delta': current_lookup.at[smooth_name, 'finite_pct'] - current_lookup.at[raw_name, 'finite_pct'],
            'avg_turnover_proxy_delta': current_lookup.at[smooth_name, 'avg_turnover_proxy'] - current_lookup.at[raw_name, 'avg_turnover_proxy'],
            'quality_status_raw': current_lookup.at[raw_name, 'status'],
            'quality_status_v4': current_lookup.at[smooth_name, 'status'],
        })
current_v3_v4_comparison = pd.DataFrame(current_comparison_rows)
print('Current v3/v4 quality diagnostics')
display(current_quality_diag)
print('Current v3 raw vs v4 smooth comparison')
display(current_v3_v4_comparison)

metadata = load_table("alpha_construction_metadata_current")
orthogonal_alpha_name = 'alpha_orthogonal_diversifier_v1_smooth'
v4_alpha_names = [
    'alpha_decay_aware_dynamic_v4_smooth',
    'alpha_rolling_ic_dynamic_v4_smooth',
    'alpha_regime_blend_dynamic_v4_smooth',
    'alpha_hybrid_adaptive_v4_smooth',
]
current_sleeve_correlation_report = pd.DataFrame()
if not corr.empty:
    current_sleeve_correlation_report = corr.loc[
        (corr['alpha_name_1'].eq(orthogonal_alpha_name) & corr['alpha_name_2'].isin(v4_alpha_names))
        | (corr['alpha_name_2'].eq(orthogonal_alpha_name) & corr['alpha_name_1'].isin(v4_alpha_names))
    ].copy()
    if not current_sleeve_correlation_report.empty:
        current_sleeve_correlation_report['comparison_alpha'] = np.where(
            current_sleeve_correlation_report['alpha_name_1'].eq(orthogonal_alpha_name),
            current_sleeve_correlation_report['alpha_name_2'],
            current_sleeve_correlation_report['alpha_name_1'],
        )
        current_sleeve_correlation_report['abs_correlation'] = pd.to_numeric(
            current_sleeve_correlation_report['correlation'],
            errors='coerce',
        ).abs()
        current_sleeve_correlation_report['orthogonality_flag'] = np.where(
            current_sleeve_correlation_report['abs_correlation'].gt(0.70),
            'ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70',
            'ORTHOGONALITY_PRESERVED_ABS_CORR_LE_0P70',
        )
        current_sleeve_correlation_report = current_sleeve_correlation_report.sort_values('abs_correlation', ascending=False)

print('Current alpha sleeve metadata')
display(metadata[[
    'alpha_name',
    'alpha_sleeve',
    'source_signal_names',
    'source_signal_horizons',
    'source_diversity_groups',
    'source_orthogonal_version',
    'smoothing_window',
    'rebalance_frequency',
    'turnover_control_enabled',
    'source_alpha_version',
]].sort_values('alpha_name'))
print('Current orthogonal sleeve correlation report')
display(current_sleeve_correlation_report)


source_role
WATCHLIST_DIVERSIFIER    3
DIVERSITY_SELECTED       1
Name: count, dtype: int64

signal_family
volatility_structure              3
cross_sectional_relative_value    1
Name: count, dtype: int64

,alpha_name,finite_pct,missing_pct,max_abs_alpha,avg_turnover_proxy,n_dates,n_tickers,first_valid_date,last_valid_date,status,quality_notes,run_id,alpha_construction_version
0,alpha_decay_aware_dynamic_v3,0.995785,0.004215,3.000000,10.346504,2098,478,2018-03-29,2026-05-07,REJECTED_ALPHA_CONSTRUCTION,"Fails construction coverage, scale, or turnove...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
4,alpha_decay_aware_dynamic_v4_smooth,0.995625,0.004375,3.000000,1.737177,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
3,alpha_hybrid_adaptive_v3,0.995376,0.004624,3.000000,10.286685,2098,478,2018-03-29,2026-05-07,REJECTED_ALPHA_CONSTRUCTION,"Fails construction coverage, scale, or turnove...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
7,alpha_hybrid_adaptive_v4_smooth,0.995486,0.004514,3.000000,1.738334,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
8,alpha_orthogonal_diversifier_v1_smooth,0.991994,0.008006,3.000000,1.683714,2098,478,2018-04-06,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
9,alpha_orthogonal_diversifier_v2_score_weighted...,0.995255,0.004745,2.811819,1.946872,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
1,alpha_regime_blend_dynamic_v3,0.995785,0.004215,3.000000,10.559577,2098,478,2018-03-29,2026-05-07,REJECTED_ALPHA_CONSTRUCTION,"Fails construction coverage, scale, or turnove...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
5,alpha_regime_blend_dynamic_v4_smooth,0.995625,0.004375,3.000000,1.746788,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
2,alpha_rolling_ic_dynamic_v3,0.995376,0.004624,3.000000,9.865850,2098,478,2018-03-29,2026-05-07,REJECTED_ALPHA_CONSTRUCTION,"Fails construction coverage, scale, or turnove...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
6,alpha_rolling_ic_dynamic_v4_smooth,0.995486,0.004514,3.000000,1.743410,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4


,alpha_name,mean_abs_alpha,alpha_std,max_abs_alpha,avg_turnover_proxy,median_turnover_proxy,max_turnover_proxy,turnover_risk_flag,finite_pct,n_dates,n_tickers,dynamic_alpha_flag,avg_effective_n_components,max_component_weight,run_id,alpha_construction_version
0,alpha_decay_aware_dynamic_v3,0.879096,1.096330,3.000000,10.346504,9.989610,24.304348,HIGH_TURNOVER_RISK,0.995785,2098,478,1,3.802192,0.294920,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
4,alpha_decay_aware_dynamic_v4_smooth,0.761009,0.922627,3.000000,1.737177,0.306667,10.250000,LOW_TURNOVER_RISK,0.995625,2098,478,1,3.802192,0.294920,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
3,alpha_hybrid_adaptive_v3,0.884657,1.101829,3.000000,10.286685,9.965000,23.730000,HIGH_TURNOVER_RISK,0.995376,2098,478,1,3.802192,0.294920,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
7,alpha_hybrid_adaptive_v4_smooth,0.764794,0.928293,3.000000,1.738334,0.309365,10.075000,LOW_TURNOVER_RISK,0.995486,2098,478,1,3.802192,0.294920,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
8,alpha_orthogonal_diversifier_v1_smooth,0.359434,0.453302,3.000000,1.683714,0.321608,9.922206,LOW_TURNOVER_RISK,0.991994,2098,478,0,1.000000,1.000000,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
9,alpha_orthogonal_diversifier_v2_score_weighted...,0.320720,0.392292,2.811819,1.946872,0.281221,11.083333,MODERATE_TURNOVER_RISK,0.995255,2098,478,0,3.874684,0.295423,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
1,alpha_regime_blend_dynamic_v3,0.885105,1.102477,3.000000,10.559577,10.196667,25.567568,HIGH_TURNOVER_RISK,0.995785,2098,478,1,3.724715,0.401788,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
5,alpha_regime_blend_dynamic_v4_smooth,0.766513,0.930999,3.000000,1.746788,0.308725,10.196667,LOW_TURNOVER_RISK,0.995625,2098,478,1,3.724715,0.401788,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
2,alpha_rolling_ic_dynamic_v3,0.882067,1.098439,3.000000,9.865850,9.451050,31.979167,HIGH_TURNOVER_RISK,0.995376,2098,478,1,4.383750,0.350000,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
6,alpha_rolling_ic_dynamic_v4_smooth,0.778661,0.955571,3.000000,1.743410,0.296296,11.421141,LOW_TURNOVER_RISK,0.995486,2098,478,1,4.383750,0.350000,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4


,count,mean,min,max
alpha_name,,,,
alpha_decay_aware_dynamic_v3,8392,0.250000,0.153123,0.294920
alpha_hybrid_adaptive_v3,8392,0.250000,0.153123,0.294920
alpha_orthogonal_diversifier_v1_smooth,2098,1.000000,1.000000,1.000000
alpha_orthogonal_diversifier_v2_score_weighted_smooth,8392,0.250000,0.200855,0.295423
alpha_regime_blend_dynamic_v3,8392,0.250000,0.111935,0.401788
alpha_rolling_ic_dynamic_v3,8392,0.202367,0.000000,0.350000


...

,alpha_name_raw_v3,alpha_name_v4,finite_pct_delta,avg_turnover_proxy_delta,quality_status_raw,quality_status_v4
0,alpha_decay_aware_dynamic_v3,alpha_decay_aware_dynamic_v4_smooth,-0.00016,-8.609327,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION
1,alpha_rolling_ic_dynamic_v3,alpha_rolling_ic_dynamic_v4_smooth,0.00011,-8.122439,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION
2,alpha_regime_blend_dynamic_v3,alpha_regime_blend_dynamic_v4_smooth,-0.00016,-8.812789,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION
3,alpha_hybrid_adaptive_v3,alpha_hybrid_adaptive_v4_smooth,0.00011,-8.548351,REJECTED_ALPHA_CONSTRUCTION,APPROVED_FOR_ALPHA_VALIDATION


Current alpha sleeve metadata


,alpha_name,alpha_sleeve,source_signal_names,source_signal_horizons,source_diversity_groups,source_orthogonal_version,smoothing_window,rebalance_frequency,turnover_control_enabled,source_alpha_version
0,alpha_decay_aware_dynamic_v3,DECAY_STABILITY,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,NaN,NaN,0,v3/raw
4,alpha_decay_aware_dynamic_v4_smooth,DECAY_STABILITY,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,10.0,5.0,1,v4/smooth
3,alpha_hybrid_adaptive_v3,CORE_REGIME,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,NaN,NaN,0,v3/raw
7,alpha_hybrid_adaptive_v4_smooth,CORE_REGIME,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,10.0,5.0,1,v4/smooth
8,alpha_orthogonal_diversifier_v1_smooth,ORTHOGONAL_DIVERSIFIER,vol_of_vol_20,10,ORTHOGONAL_DIVERSIFIER_SELECTED,phase2_orthogonal_signals_v2,10.0,5.0,1,v1/orthogonal_smooth
9,alpha_orthogonal_diversifier_v2_score_weighted...,ORTHOGONAL_DIVERSIFIER,"vol_surprise_20_60,price_impact_proxy_20,range...","20,20,20,5",RULE_BASED_ORTHOGONAL_CANDIDATE,v2_rule_based_candidate_screen,10.0,5.0,1,v2/orthogonal_score_weighted_smooth
1,alpha_regime_blend_dynamic_v3,CORE_REGIME,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,NaN,NaN,0,v3/raw
5,alpha_regime_blend_dynamic_v4_smooth,CORE_REGIME,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,10.0,5.0,1,v4/smooth
2,alpha_rolling_ic_dynamic_v3,CORE_REGIME,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,NaN,NaN,0,v3/raw
6,alpha_rolling_ic_dynamic_v4_smooth,CORE_REGIME,"vol_of_vol_20,range_expansion_failure_5,residu...","10,20,20,20",ORTHOGONAL_DIVERSIFIER_SELECTED,None,10.0,5.0,1,v4/smooth


Current orthogonal sleeve correlation report


,alpha_name_1,alpha_name_2,correlation,run_id,alpha_construction_version,comparison_alpha,abs_correlation,orthogonality_flag
48,alpha_decay_aware_dynamic_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,0.834848,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_decay_aware_dynamic_v4_smooth,0.834848,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
84,alpha_orthogonal_diversifier_v1_smooth,alpha_decay_aware_dynamic_v4_smooth,0.834848,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_decay_aware_dynamic_v4_smooth,0.834848,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
78,alpha_hybrid_adaptive_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,0.830409,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_hybrid_adaptive_v4_smooth,0.830409,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
87,alpha_orthogonal_diversifier_v1_smooth,alpha_hybrid_adaptive_v4_smooth,0.830409,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_hybrid_adaptive_v4_smooth,0.830409,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
58,alpha_regime_blend_dynamic_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,0.820274,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_regime_blend_dynamic_v4_smooth,0.820274,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
85,alpha_orthogonal_diversifier_v1_smooth,alpha_regime_blend_dynamic_v4_smooth,0.820274,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_regime_blend_dynamic_v4_smooth,0.820274,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
68,alpha_rolling_ic_dynamic_v4_smooth,alpha_orthogonal_diversifier_v1_smooth,0.747806,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_rolling_ic_dynamic_v4_smooth,0.747806,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
86,alpha_orthogonal_diversifier_v1_smooth,alpha_rolling_ic_dynamic_v4_smooth,0.747806,phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4,alpha_rolling_ic_dynamic_v4_smooth,0.747806,ORTHOGONALITY_WEAK_ABS_CORR_GT_0P70
